# FrozenLake Value Iteration

## 实验目标

本实验使用 `Value Iteration` 求解 FrozenLake，目标是展示：当环境的状态转移结构已知时，可以不依赖反复试错训练，而是直接通过动态规划迭代最优价值函数，再反推出最优策略。这个实验和 `01` 中的 `Q-Learning` 正好形成鲜明对照。

## 为什么这里选择 Value Iteration

FrozenLake 是一个小型离散环境，状态和动作都有限，而且底层环境对象可以提供完整的转移结构。在这种前提下，使用 `Value Iteration` 很自然，因为它可以直接利用环境模型做规划，而不是像 `Q-Learning` 那样完全依赖交互经验累积。

因此，这个实验的重点不是现代工程训练技巧，而是用最清楚的方式展示：已知模型时，如何通过 Bellman 最优性迭代直接得到策略。

## 设备与并行说明

- 本实验默认使用 `CPU`
- 不使用 `CUDA` 或 `MPS`
- 不使用并行环境

原因是 `Value Iteration` 本身只是对有限状态空间做数值迭代，规模很小，没有必要引入设备加速或并行采样。

In [ ]:
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
ENV_ID = "FrozenLake-v1"
GAMMA = 0.99
THETA = 1e-10
MAX_ITERATIONS = 1000
EVAL_EPISODES = 5000
MAX_STEPS = 100
SEED = 42

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

env = gym.make(ENV_ID, is_slippery=True)
num_states = env.observation_space.n
num_actions = env.action_space.n
transition_model = env.unwrapped.P

## 参数选择说明

- `GAMMA = 0.99`：终点奖励是长期目标，因此保持较高折扣因子
- `THETA = 1e-10`：价值收敛阈值，足够小，便于在这个小环境中逼近稳定解
- `MAX_ITERATIONS = 1000`：提供安全上限，防止理论上没问题但实现上意外死循环
- `EVAL_EPISODES = 5000`：策略评估阶段使用较多回合，保证统计结果稳定

这里的参数重点不在于训练速度，而在于让价值迭代收敛结果足够清楚。

In [ ]:
def one_step_lookahead(state, values, gamma):
    action_values = np.zeros(num_actions, dtype=np.float64)
    for action in range(num_actions):
        for prob, next_state, reward, terminated in transition_model[state][action]:
            action_values[action] += prob * (reward + gamma * values[next_state] * (1 - int(terminated)))
    return action_values

In [ ]:
values = np.zeros(num_states, dtype=np.float64)
delta_history = []

for iteration in tqdm(range(MAX_ITERATIONS), desc="Value iteration"):
    delta = 0.0
    new_values = values.copy()

    for state in range(num_states):
        action_values = one_step_lookahead(state, values, GAMMA)
        best_value = np.max(action_values)
        delta = max(delta, abs(best_value - values[state]))
        new_values[state] = best_value

    values = new_values
    delta_history.append(delta)

    if delta < THETA:
        break

num_iterations = len(delta_history)
values

In [ ]:
policy = np.zeros(num_states, dtype=np.int64)
for state in range(num_states):
    action_values = one_step_lookahead(state, values, GAMMA)
    policy[state] = int(np.argmax(action_values))

policy

In [ ]:
episode_rewards = []
episode_lengths = []
successes = []

for episode in tqdm(range(EVAL_EPISODES), desc="Policy evaluation"):
    state, info = env.reset(seed=SEED + episode)
    total_reward = 0.0

    for step in range(MAX_STEPS):
        action = policy[state]
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if terminated or truncated:
            successes.append(int(reward > 0))
            episode_lengths.append(step + 1)
            break
    else:
        successes.append(0)
        episode_lengths.append(MAX_STEPS)

    episode_rewards.append(total_reward)

env.close()

In [ ]:
success_rate = float(np.mean(successes))
avg_reward = float(np.mean(episode_rewards))
avg_length = float(np.mean(episode_lengths))

metrics = pd.DataFrame(
    {
        "metric": [
            "num_iterations",
            "final_delta",
            "success_rate",
            "average_reward",
            "average_episode_length",
        ],
        "value": [
            num_iterations,
            delta_history[-1],
            success_rate,
            avg_reward,
            avg_length,
        ],
    }
)
metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(delta_history, color="#c44e52")
axes[0].set_title("Value Iteration Delta")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Max Value Change")
axes[0].set_yscale("log")

value_grid = values.reshape(4, 4)
im = axes[1].imshow(value_grid, cmap="Blues")
axes[1].set_title("State Value Heatmap")
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f"{value_grid[i, j]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

action_symbols = np.array(["L", "D", "R", "U"])
policy_grid = action_symbols[policy].reshape(4, 4)
axes[2].imshow(np.zeros((4, 4)), cmap="Greys", vmin=0, vmax=1)
axes[2].set_title("Greedy Policy Map")
for i in range(4):
    for j in range(4):
        axes[2].text(j, i, policy_grid[i, j], ha="center", va="center", color="black", fontsize=14)
axes[2].set_xticks(range(4))
axes[2].set_yticks(range(4))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "value_iteration_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
metrics.to_csv(RESULTS_DIR / "value_iteration_metrics.csv", index=False)
pd.DataFrame({"delta": delta_history}).to_csv(RESULTS_DIR / "value_iteration_delta_history.csv", index=False)
pd.DataFrame(values.reshape(4, 4)).to_csv(RESULTS_DIR / "value_iteration_state_values.csv", index=False)
pd.DataFrame(policy.reshape(4, 4)).to_csv(RESULTS_DIR / "value_iteration_policy.csv", index=False)

print(f"Iterations until convergence: {num_iterations}")
print(f"Final delta: {delta_history[-1]:.12f}")
print(f"Success rate: {success_rate:.4f}")
print(f"Average reward: {avg_reward:.4f}")
print(f"Average episode length: {avg_length:.2f}")
print(f"Saved results to: {RESULTS_DIR.resolve()}")

## 结果解读

这个实验最重要的不是训练曲线，而是规划思路本身。重点看三件事：

- `delta` 是否快速下降到很小，说明价值迭代已经收敛
- 状态价值热力图是否在接近目标区域呈现更高价值
- 贪心策略图是否形成一条稳定、可解释的前进方向

如果你的运行结果类似下面这种形式：

- `Iterations until convergence: 571`
- `Final delta: 0.000000000098`
- `Success rate: 0.7314`
- `Average reward: 0.7314`
- `Average episode length: 44.54`

那么它的含义是：

- 价值迭代在 571 轮更新后达到收敛阈值
- 最终 `delta` 只剩大约 `9.8e-11`，说明值函数数值上已经非常稳定
- 最终策略在评估阶段大约有 `73.14%` 的概率能成功到达终点
- 由于 FrozenLake 在 `is_slippery=True` 时存在随机滑动，最优策略也不可能保证每次都成功，因此成功率不是 100% 是正常现象
- 平均回报和成功率相同，是因为这个环境里成功通常记为 1，失败记为 0
- 平均回合长度约为 44.54 步，说明即使策略已经合理，在随机滑动影响下，很多回合仍然会出现绕路或额外步数

与 `Q-Learning` 不同，`Value Iteration` 不需要通过反复交互逐渐学值，而是直接利用环境模型进行规划。这正是本实验与 `01` 的核心对照价值。